# Metadata

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

ROOT = next(
  path
  for path in (Path.cwd().resolve(),
  *Path.cwd().resolve().parents)
  if (path / "pyproject.toml").exists()
)

DATA_DIR = ROOT / "data" / "symile-mimic"
NPY_DIR = DATA_DIR / "data_npy"
RAW_METADATA_PATH = DATA_DIR / "symile_mimic_data.csv"

print(ROOT)
print(DATA_DIR)

/mnt/disk1/home/cgiovannoni/isic24-dum
/mnt/disk1/home/cgiovannoni/isic24-dum/data/symile-mimic


In [3]:
raw_df = pd.read_csv(RAW_METADATA_PATH)

print("Raw shape:", raw_df.shape)
display(raw_df.head(2))

assert raw_df["hadm_id"].is_unique
assert raw_df["hospital_expire_flag"].isin([0, 1]).all()

print("\nMortality distribution:")

display(raw_df["hospital_expire_flag"].value_counts(normalize=True))

Raw shape: (11622, 94)


,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,race,hospital_expire_flag,...,50804,50818,52075,52073,52074,52069,51133,50910,52135,labs_all_nan
0,10001176,23334588,2186-11-29 03:56:00,2186-12-02 15:00:00,NaN,EU OBSERVATION,EMERGENCY ROOM,NaN,WHITE,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,10001401,24818636,2131-07-30 21:40:00,2131-08-04 14:10:00,NaN,OBSERVATION ADMIT,EMERGENCY ROOM,SKILLED NURSING FACILITY,WHITE,0,...,NaN,NaN,4.97,0.05,0.89,0.05,1.58,NaN,1.4,0



Mortality distribution:


hospital_expire_flag
0    0.907503
1    0.092497
Name: proportion, dtype: float64

In [4]:
train_hadm_ids = np.load(
  NPY_DIR / "train" / "hadm_id_train.npy",
  mmap_mode="r",
)

val_hadm_ids = np.load(
  NPY_DIR / "val" / "hadm_id_val.npy",
  mmap_mode="r",
)

test_hadm_ids = np.load(
  NPY_DIR / "test" / "hadm_id_test.npy",
  mmap_mode="r",
)

test_retrieval_labels = np.load(
  NPY_DIR / "test" / "label_test.npy",
  mmap_mode="r",
)

test_positive_mask = test_retrieval_labels == 1
test_positive_hadm_ids =test_hadm_ids[test_positive_mask]

assert len(test_positive_hadm_ids) == len(set(test_positive_hadm_ids))

print("Train CXR:", len(train_hadm_ids))
print("Val CXR:", len(val_hadm_ids))
print("Unique test CXR:", len(test_positive_hadm_ids))

Train CXR: 10000
Val CXR: 750
Unique test CXR: 464


In [5]:
def make_image_index(hadm_ids, source_split,
row_indices=None):
  if row_indices is None:
      row_indices = np.arange(len(hadm_ids))

  return pd.DataFrame(
      {
          "hadm_id": np.asarray(hadm_ids,
          dtype="int64"),
          "source_split": source_split,
          "source_row": np.asarray(row_indices,
          dtype="int64"),
      }
  )

image_index = pd.concat(
  [
      make_image_index(train_hadm_ids, "train"),
      make_image_index(val_hadm_ids, "val"),
      make_image_index(
          test_positive_hadm_ids,
          "test",
          np.flatnonzero(test_positive_mask),
      ),
  ],
  ignore_index=True,
)

assert image_index["hadm_id"].is_unique

print("Total CXR available:", len(image_index))
display(image_index.head())

Total CXR available: 11214


,hadm_id,source_split,source_row
0,25296721,train,0
1,26188372,train,1
2,20652011,train,2
3,26177106,train,3
4,22280697,train,4


In [7]:
cohort_df = raw_df.merge(
  image_index,
  on="hadm_id",
  how="inner",
  validate="one_to_one",
)

assert cohort_df["hadm_id"].is_unique
assert len(cohort_df) == len(image_index)

print("Cohort shape:", cohort_df.shape)
print("Patients:", cohort_df["subject_id"].nunique())
print("Admissions:", cohort_df["hadm_id"].nunique())

print("\nMortality distribution:")

display(cohort_df["hospital_expire_flag"].value_counts(normalize=True))

Cohort shape: (11214, 96)
Patients: 9573
Admissions: 11214

Mortality distribution:


hospital_expire_flag
0    0.906545
1    0.093455
Name: proportion, dtype: float64

In [9]:
SYMLE_CODE_DIR = DATA_DIR / "code"
sys.path.insert(0, str(SYMLE_CODE_DIR))

from constants import LABS

LAB_COLUMNS = list(LABS.keys())

FEATURES = {
  "identifiers": [
      "subject_id",
      "hadm_id",
      "source_split",
      "source_row",
  ],
  "target": "hospital_expire_flag",
  "demographic": [
      "age",
      "gender",
  ],
  "admission_metadata": [
      "admission_type",
      "admission_location",
      "cxr_ViewPosition",
  ],
  "labs_raw": LAB_COLUMNS,
  "forbidden": [
      "dischtime",
      "deathtime",
      "dod",
      "discharge_location",
      "ecg_study_id",
      "ecg_time",
      "ecg_path",
  ],
}

allowed_columns = (
  FEATURES["identifiers"]
  + [FEATURES["target"]]
  + FEATURES["demographic"]
  + FEATURES["admission_metadata"]
  + FEATURES["labs_raw"]
)

missing_columns = set(allowed_columns) - set(cohort_df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

print("Raw model features:", len(
  FEATURES["demographic"]
  + FEATURES["admission_metadata"]
  + FEATURES["labs_raw"]
))

Raw model features: 55


In [10]:
splitter = StratifiedGroupKFold(
  n_splits=5,
  shuffle=True,
  random_state=42,
)

train_idx, test_idx = next(
  splitter.split(
      X=cohort_df,
      y=cohort_df["hospital_expire_flag"],
      groups=cohort_df["subject_id"],
  )
)

train_df = cohort_df.iloc[train_idx].copy()
test_df = cohort_df.iloc[test_idx].copy()

assert set(train_df["subject_id"]).isdisjoint(test_df["subject_id"])
assert set(train_df["hadm_id"]).isdisjoint(test_df["hadm_id"])

print("Train shape:", train_df.shape)

display(train_df["hospital_expire_flag"].value_counts(normalize=True))

print("Test shape:", test_df.shape)

display(test_df["hospital_expire_flag"].value_counts(normalize=True))

Train shape: (8972, 96)


hospital_expire_flag
0    0.906598
1    0.093402
Name: proportion, dtype: float64

Test shape: (2242, 96)


hospital_expire_flag
0    0.906334
1    0.093666
Name: proportion, dtype: float64

In [12]:
PROCESSED_DIR = DATA_DIR / "processed"

LAB_RENAME_MAP = {
  lab_id: f"lab_{lab_id}"
  for lab_id in FEATURES["labs_raw"]
}

MODEL_COLUMNS = (
  FEATURES["demographic"]
  + FEATURES["admission_metadata"]
  + FEATURES["labs_raw"]
)

def prepare_metadata_for_export(df):
  export_df = df[
      [
          "hadm_id",
          "subject_id",
          FEATURES["target"],
          "source_split",
          "source_row",
      ]
      + MODEL_COLUMNS
  ].copy()

  export_df = export_df.rename(
      columns={
          "hadm_id": "admission_id",
          "subject_id": "patient_id",
          FEATURES["target"]: "target",
          **LAB_RENAME_MAP,
      }
  )

  export_df["image_id"] = (
      "hadm_" + export_df["admission_id"].astype(str)
  )

  export_df["patient_id"] = export_df["patient_id"].astype(str)
  export_df["target"] = export_df["target"].astype("int8")

  ordered_columns = [
      "image_id",
      "patient_id",
      "admission_id",
      "target",
      "source_split",
      "source_row",
      *FEATURES["demographic"],
      *FEATURES["admission_metadata"],
      *LAB_RENAME_MAP.values(),
  ]

  return export_df[ordered_columns]

train_metadata = prepare_metadata_for_export(train_df)
test_metadata = prepare_metadata_for_export(test_df)

display(train_metadata.head(2))
print(train_metadata.shape, test_metadata.shape)

,image_id,patient_id,admission_id,target,source_split,source_row,age,gender,admission_type,admission_location,...,lab_50821,lab_50804,lab_50818,lab_52075,lab_52073,lab_52074,lab_52069,lab_51133,lab_50910,lab_52135
0,hadm_23334588,10001176,23334588,0,train,9292,64,F,EU OBSERVATION,EMERGENCY ROOM,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,hadm_24818636,10001401,24818636,0,train,4325,89,F,OBSERVATION ADMIT,EMERGENCY ROOM,...,NaN,NaN,NaN,4.97,0.05,0.89,0.05,1.58,NaN,1.4


(8972, 61) (2242, 61)


In [13]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_metadata.to_parquet(
  PROCESSED_DIR / "train-metadata.parquet",
  index=False,
  engine="pyarrow",
)

test_metadata.to_parquet(
  PROCESSED_DIR / "test-metadata.parquet",
  index=False,
  engine="pyarrow",
)

print("Saved:", PROCESSED_DIR / "train-metadata.parquet")
print("Saved:", PROCESSED_DIR / "test-metadata.parquet")

Saved: /mnt/disk1/home/cgiovannoni/isic24-dum/data/symile-mimic/processed/train-metadata.parquet
Saved: /mnt/disk1/home/cgiovannoni/isic24-dum/data/symile-mimic/processed/test-metadata.parquet


In [14]:
cxr_arrays = {
  "train": np.load(
      NPY_DIR / "train" / "cxr_train.npy",
      mmap_mode="r",
  ),
  "val": np.load(
      NPY_DIR / "val" / "cxr_val.npy",
      mmap_mode="r",
  ),
  "test": np.load(
      NPY_DIR / "test" / "cxr_test.npy",
      mmap_mode="r",
  ),
}

for split_name, array in cxr_arrays.items():
  print(split_name, array.shape, array.dtype)

train (10000, 3, 320, 320) float32
val (750, 3, 320, 320) float32
test (4640, 3, 320, 320) float32


In [15]:
### Cella 11 — Conversione CXR tensor → JPEG bytes

import io
import h5py

from PIL import Image
from tqdm.auto import tqdm

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406],
dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225],
dtype=np.float32)

def cxr_tensor_to_jpeg_bytes(cxr_tensor, quality=100):
  """
  Converte un tensor Symile CxHxW, ImageNet-
  normalized,
  in un JPEG RGB leggibile dal loader HDF5 esistente.
  """
  image = np.asarray(cxr_tensor, dtype=np.float32)

  assert image.shape == (3, 320, 320), image.shape

  image = np.moveaxis(image, 0, -1)
  image = image * IMAGENET_STD + IMAGENET_MEAN
  image = np.clip(image * 255.0, 0,
  255).round().astype(np.uint8)

  pil_image = Image.fromarray(image, mode="RGB")

  buffer = io.BytesIO()
  pil_image.save(
      buffer,
      format="JPEG",
      quality=quality,
      subsampling=0,
  )

  return buffer.getvalue()

In [16]:
def create_cxr_hdf5(metadata_df, output_path):
  with h5py.File(output_path, "w") as h5:
      h5.attrs["format"] = "jpeg_bytes"
      h5.attrs["image_size"] = "(320, 320)"
      h5.attrs["id_col"] = "image_id"
      h5.attrs["source"] = "symile-mimic data_npy"

      rows = metadata_df[
          ["image_id", "source_split", "source_row"]
      ].itertuples(index=False)

      for row in tqdm(
          rows,
          total=len(metadata_df),
          desc=f"Writing {output_path.name}",
      ):
          cxr_tensor = cxr_arrays[row.source_split]
          [row.source_row]
          jpeg_bytes = cxr_tensor_to_jpeg_bytes(cxr_tensor)

          h5.create_dataset(
              row.image_id,
              data=np.bytes_(jpeg_bytes),
          )

  print(f"Created: {output_path}")

create_cxr_hdf5(
  train_metadata,
  PROCESSED_DIR / "train-image.hdf5",
)

create_cxr_hdf5(
  test_metadata,
  PROCESSED_DIR / "test-image.hdf5",
)

Writing train-image.hdf5:   0%|          | 0/8972 [00:00<?, ?it/s]

AssertionError: (10000, 3, 320, 320)

In [ ]:
### Cella 13 — Sanity check finale

for split_name, metadata in {
  "train": train_metadata,
  "test": test_metadata,
}.items():
  hdf5_path = PROCESSED_DIR /
  f"{split_name}-image.hdf5"
  image_id = metadata["image_id"].iloc[0]

  with h5py.File(hdf5_path, "r") as h5:
      assert len(h5) == len(metadata)
      assert image_id in h5

      jpeg_bytes = h5[image_id][()].tobytes()

  image = Image.open(io.BytesIO(jpeg_bytes)).convert("RGB")

  print(
      f"{split_name}: "
      f"{len(metadata)} metadata rows, "
      f"image {image_id}, size={image.size}"
  )

assert set(train_metadata["patient_id"]).isdisjoint(
  set(test_metadata["patient_id"])
)

print("All checks passed.")

In [8]:
df = df.copy()
to_drop = ['patient_age', 'illuminaton', 'diabetic_retinopathy', 'dr_icdr', 'dr_sdrg']

df.columns = map(str.lower, df.columns)

df['age_approx'] = df['patient_age']
df['sex'] = df['patient_sex'].map({1: 'male', 2: 'female'}).fillna('unknown')
df['illumination'] = df['illuminaton']
df["anatom_site_general"] = df["exam_eye"].map({1: "right_eye", 2: "left_eye"}).fillna("unknown")
df["target"] = df["diabetic_retinopathy"].astype(int)

df.drop(to_drop, axis=1, inplace=True)

In [9]:
# create train/test sets
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(n_splits=5,
                                shuffle=True,
                                random_state=42,
)

train_idx, test_idx = next(splitter.split(df,
                                          y=df["target"],
                                          groups=df["patient_id"],
                                          )
)

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

In [10]:
overlap = set(train_df["patient_id"]) & set(test_df["patient_id"])
assert len(overlap) == 0, f"Patient leakage detected: {len(overlap)} patients overlap"

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))

print("\nTrain target distribution:")
print(train_df["target"].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_df["target"].value_counts(normalize=True))

Train samples: 13012
Test samples: 3254

Train target distribution:
target
0    0.934215
1    0.065785
Name: proportion, dtype: float64

Test target distribution:
target
0    0.934235
1    0.065765
Name: proportion, dtype: float64


In [11]:
out_dir = ROOT / "data" / 'brset' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)

train_path = out_dir / "train-metadata.parquet"
test_path = out_dir / "test-metadata.parquet"

train_df.to_parquet(train_path, index=False, engine="pyarrow")
test_df.to_parquet(test_path, index=False, engine="pyarrow")

print(f"Saved train: {train_path}")
print(f"Saved test: {test_path}")

Saved train: /home/claudio/isic24-dum/data/brset/processed/train-metadata.parquet
Saved test: /home/claudio/isic24-dum/data/brset/processed/test-metadata.parquet


# Images

In [13]:
from PIL import Image
img_path = ROOT / "data" / "brset" / "images" / 'img00042.jpg'
img = Image.open(img_path)
print(img.size)
display(img)

FileNotFoundError: [Errno 2] No such file or directory: '/home/claudio/isic24-dum/data/brset/processed/images/img00042.jpg'

In [ ]:
from pathlib import Path
import io
import h5py
import numpy as np
from PIL import Image
from tqdm import tqdm


def image_to_jpeg_bytes(image_path, size=(256, 256), quality=95):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img = img.resize(size, Image.Resampling.LANCZOS)

        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=quality)

        return buffer.getvalue()


def create_hdf5_from_metadata(df, image_dir, out_filepath, id_col="image_id", size=(256, 256)):
  image_dir = Path(image_dir)
  out_filepath = Path(out_filepath)
  out_filepath.parent.mkdir(parents=True, exist_ok=True)

  with h5py.File(out_filepath, "w") as h5:
      h5.attrs["format"] = "jpeg_bytes"
      h5.attrs["image_size"] = str(size)
      h5.attrs["id_col"] = id_col

      for image_id in tqdm(df[id_col].astype(str), desc=f"Creating {out_filepath.name}"):
          image_path = image_dir / f"{image_id}.jpg"

          if not image_path.exists():
              raise FileNotFoundError(f"Missing image: {image_path}")

          jpeg_bytes = image_to_jpeg_bytes(image_path, size=size)
          h5.create_dataset(image_id, data=np.bytes_(jpeg_bytes))

  print(f"Created: {out_filepath}")

In [ ]:
# load csv
image_dir = ROOT / "data" / "brset" / "images"
processed_dir = ROOT / "data" / "brset" / "processed"

train_metadata_path = processed_dir / "train-metadata.parquet"
test_metadata_path = processed_dir / "test-metadata.parquet"

train_df = pd.read_parquet(train_metadata_path)
test_df = pd.read_parquet(test_metadata_path)

In [ ]:
create_hdf5_from_metadata(
    df=train_df,
    image_dir=image_dir,
    out_filepath=processed_dir / "train-image.hdf5",
    id_col="image_id",
    size=(512, 512),
)

create_hdf5_from_metadata(
    df=test_df,
    image_dir=image_dir,
    out_filepath=processed_dir / "test-image.hdf5",
    id_col="image_id",
    size=(512, 512),
)

In [ ]:
#sanity-check
image_id = "img00042"

with h5py.File(processed_dir / "train-image.hdf5", "r") as h5:
    file_bytes = h5[image_id][()].tobytes()

img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
print(img.size)
display(img)